In [1]:
# Import heapq for the A* priority queue.
# Import deque for the BFS used by the heuristic.
import heapq
from collections import deque

In [2]:
# CITY GRAPH CHECK

# The numbers are EDGE COSTS
# The graph is undirected, so each road is stored in both directions.

graph = {
    "Seattle": {
        "San Francisco": 678,
        "Chicago": 1737
        },

    "San Francisco": {
        "Seattle": 678,
        "Los Angeles": 348,
        "Riverside": 386
        },

    "Los Angeles": {
        "San Francisco": 348,
        "Riverside": 50,
        "Phoenix": 357
        },

    "Riverside": {
        "San Francisco": 386,
        "Los Angeles": 50,
        "Phoenix": 307,
        "Chicago": 1704
        },

    "Phoenix": {
        "Los Angeles": 357,
        "Riverside": 307,
        "Dallas": 887,
        "Houston": 1015
        },

    "Dallas": {
        "Phoenix": 887,
        "Chicago": 805,
        "Houston": 225,
        "Atlanta": 721
        },

    "Houston": {
        "Phoenix": 1015,
        "Dallas": 225,
        "Atlanta": 702,
        "Miami": 968
        },

    "Chicago": {
        "Seattle": 1737,
        "Riverside": 1704,
        "Dallas": 805,
        "Atlanta": 588,
        "Detroit": 238
        },

    "Atlanta": {
        "Dallas": 721,
        "Houston": 702,
        "Chicago": 588,
        "Washington": 543,
        "Miami": 604
        },
    "Detroit": {
        "Chicago": 238,
        "Boston": 613,
        "New York": 482,
        "Washington": 396
        },

    "Washington": {
        "Detroit": 396,
        "Atlanta": 543,
        "Philadelphia": 123,
        "Miami": 923
        },

    "Philadelphia": {
        "Washington": 123,
        "New York": 81
        },

    "New York": {
        "Detroit": 482,
        "Boston": 190,
        "Philadelphia": 81
        },

    "Boston": {
        "Detroit": 613,
        "New York": 190
        },

    "Miami": {
        "Houston": 968,
        "Atlanta": 604,
        "Washington": 923
        }
}


In [3]:
# Display the map
print("=" * 80)
print("CITY GRAPH")
print("=" * 80)
for city, neighbours in graph.items():
    print(f"\n{city}:")
    for neighbour, distance in neighbours.items():
        print(f"  -> {neighbour}: {distance}")

CITY GRAPH

Seattle:
  -> San Francisco: 678
  -> Chicago: 1737

San Francisco:
  -> Seattle: 678
  -> Los Angeles: 348
  -> Riverside: 386

Los Angeles:
  -> San Francisco: 348
  -> Riverside: 50
  -> Phoenix: 357

Riverside:
  -> San Francisco: 386
  -> Los Angeles: 50
  -> Phoenix: 307
  -> Chicago: 1704

Phoenix:
  -> Los Angeles: 357
  -> Riverside: 307
  -> Dallas: 887
  -> Houston: 1015

Dallas:
  -> Phoenix: 887
  -> Chicago: 805
  -> Houston: 225
  -> Atlanta: 721

Houston:
  -> Phoenix: 1015
  -> Dallas: 225
  -> Atlanta: 702
  -> Miami: 968

Chicago:
  -> Seattle: 1737
  -> Riverside: 1704
  -> Dallas: 805
  -> Atlanta: 588
  -> Detroit: 238

Atlanta:
  -> Dallas: 721
  -> Houston: 702
  -> Chicago: 588
  -> Washington: 543
  -> Miami: 604

Detroit:
  -> Chicago: 238
  -> Boston: 613
  -> New York: 482
  -> Washington: 396

Washington:
  -> Detroit: 396
  -> Atlanta: 543
  -> Philadelphia: 123
  -> Miami: 923

Philadelphia:
  -> Washington: 123
  -> New York: 81

New York:
 

In [4]:
# Show Basic graph information.
num_of_cities = len(graph)
total_entries = sum(len(neighbours) for neighbours in graph.values())
num_of_connections = total_entries // 2
minimum_edge_distance = min(
    distance
    for neighbours in graph.values()
    for distance in neighbours.values())

print("\n" + "=" * 80)
print("GRAPH INFORMATION")
print("=" * 80)
print(f"Number of cities: {num_of_cities}")
print(f"Number of unique connections: {num_of_connections}")
print(f"Minimum edge distance: {minimum_edge_distance}")


GRAPH INFORMATION
Number of cities: 15
Number of unique connections: 26
Minimum edge distance: 50


In [5]:
# GRAPH VALIDATION

# Check:
# 1. Every neighbour is a valid city.
# 2. Every road has a reverse connection.
# 3. Both directions have the same distance.
# 4. Every distance is positive.

def validate_graph(graph):
    valid = True
    for city, neighbours in graph.items():
        for neighbour, distance in neighbours.items():
            if neighbour not in graph:
                print(f"ERROR: {neighbour} is not a city in the graph.")
                valid = False
                continue
            if city not in graph[neighbour]:
                print(f"ERROR: Missing reverse connection: {neighbour} -> {city}")
                valid = False
                continue
            if graph[neighbour][city] != distance:
                print(f"ERROR: Distance mismatch between {city} and {neighbour}")
                valid = False
            if distance <= 0:
                print(f"ERROR: Edge distance must be positive: {city} -> {neighbour}")
                valid = False

    if valid:
        print("Graph validation successful.")
        print("All connections are bidirectional and distances are consistent.")
    return valid
validate_graph(graph)


Graph validation successful.
All connections are bidirectional and distances are consistent.


True

## Heuristic \(h(n)\)

The map gives road distances, but it does not give a heuristic value for every city.

Therefore construct a lower-bound heuristic from the graph.

1. Find the minimum edge distance. Here it is 50.
2. Use BFS to find the minimum **number of remaining edges** to the goal.
3. Calculate:


In [6]:
# HEURISTIC h(n) function

def heuristic(city, goal, graph):
    if city == goal:
        return 0

    queue = deque([(city, 0)])
    visited = {city}

    while queue:
        current_city, edge_count = queue.popleft()
        for neighbour in graph[current_city]:
            if neighbour == goal:
                return (edge_count + 1) * minimum_edge_distance
            if neighbour not in visited:
                visited.add(neighbour)
                queue.append((neighbour, edge_count + 1))

    return float("inf")


In [7]:
# Heuristic h(n) function testing.

test_pairs = [
    ("Los Angeles", "New York"),
    ("Seattle", "Los Angeles"),
    ("Chicago", "Miami")
]

print("=" * 80)
print("HEURISTIC EXAMPLES")
print("=" * 80)

for city, goal in test_pairs:
    print(f"h({city} -> {goal}) = {heuristic(city, goal, graph)}")


HEURISTIC EXAMPLES
h(Los Angeles -> New York) = 200
h(Seattle -> Los Angeles) = 100
h(Chicago -> Miami) = 100


## A* Search

For every city:

f(n)=g(n)+h(n)

The priority queue always gives us the candidate with the smallest `f(n)`.

- `g(n)` = actual distance travelled so far.
- `h(n)` = estimated remaining distance.
- `f(n)` = estimated total distance.

The algorithm also stores the cheapest known `g(n)` for each city.


In [8]:
# A* SEARCH

def a_star_search(graph, start, goal):
    priority_queue = []
    start_g = 0
    start_h = heuristic(start, goal, graph)
    start_f = start_g + start_h
    heapq.heappush(
        priority_queue,
        (start_f, start_g, start, [start])
    )
    best_g = {start: 0}
    steps = []
    while priority_queue:
        # Choose the candidate with the smallest f(n).
        f_cost, g_cost, current_city, path = heapq.heappop(priority_queue)
        # Ignore an outdated queue entry.
        if g_cost != best_g.get(current_city):
            continue

        h_cost = f_cost - g_cost

        steps.append(
            {
            "city": current_city,
            "g(n)": g_cost,
            "h(n)": h_cost,
            "f(n)": f_cost,
            "path": path.copy()
            })

        if current_city == goal:
            return path, g_cost, steps

        for neighbour, distance in graph[current_city].items():
            new_g = g_cost + distance

            # Check is it cheaper than the best route previously found to this neighbour.
            if new_g < best_g.get(neighbour, float("inf")):
                best_g[neighbour] = new_g
                new_h = heuristic(neighbour, goal, graph)
                # A*  function: f(n) = g(n) + h(n)
                new_f = new_g + new_h
                new_path = path + [neighbour]
                heapq.heappush(
                    priority_queue,
                    (new_f, new_g, neighbour, new_path)
                )

    return None, None, steps


In [9]:
# USER INPUT HANDLING

def get_city_input(message, graph):
    city_lookup = {city.lower(): city for city in graph}

    while True:
        user_input = input(message).strip()
        if not user_input:
            print("Error: City name cannot be empty.")
            continue
        if user_input.lower() in city_lookup:
            return city_lookup[user_input.lower()]

        print("Error: City not found.")
        print("Available cities:")
        print(", ".join(graph.keys()))


In [10]:
# RUN A* CITY NAVIGATION

print("=" * 80)
print("A* CITY NAVIGATION PROGRAM")
print("=" * 80)

print("\nAvailable cities:")
print(", ".join(graph.keys()))
print()

while True:
    default_choice = input("Default route from Seattle to Miami? (Y/N): ").strip().lower()

    if default_choice == "y":
        start_city = "Seattle"
        goal_city = "Miami"
        break

    elif default_choice == "n":
        start_city = get_city_input("Enter current city: ", graph)
        goal_city = get_city_input("Enter goal city: ", graph)
        break
    else:
        print("Please enter Y or N.")

print(f"\nCurrent city: {start_city}")
print(f"Goal city: {goal_city}")

A* CITY NAVIGATION PROGRAM

Available cities:
Seattle, San Francisco, Los Angeles, Riverside, Phoenix, Dallas, Houston, Chicago, Atlanta, Detroit, Washington, Philadelphia, New York, Boston, Miami

Default route from Seattle to Miami? (Y/N): y

Current city: Seattle
Goal city: Miami


In [11]:
# DISPLAY FINAL RESULT

path, cost, steps = a_star_search(graph, start_city, goal_city)

if path is not None:
    print("\n" + "=" * 80)
    print("SEARCH RESULT")
    print("=" * 80)
    print("\nShortest path:")
    print(" -> ".join(path))
    print("\nCities along the shortest path:")

    total_distance = 0

    for i in range(len(path)):
        current_city = path[i]
        print(f"{i + 1}. {current_city}")

        if i < len(path) - 1:
            next_city = path[i + 1]
            distance = graph[current_city][next_city]
            total_distance += distance
            print(f"      Distance to {next_city}: {distance}")

    print(f"\nTotal distance / cost: {total_distance}")

else:
    print("\nNo path exists between the selected cities.")



SEARCH RESULT

Shortest path:
Seattle -> Chicago -> Atlanta -> Miami

Cities along the shortest path:
1. Seattle
      Distance to Chicago: 1737
2. Chicago
      Distance to Atlanta: 588
3. Atlanta
      Distance to Miami: 604
4. Miami

Total distance / cost: 2929


In [12]:
# ============================================================
# A* EXPANSION DETAILS
# ============================================================
# This table makes f(n) = g(n) + h(n) visible.

print("\n" + "=" * 80)
print("A* EXPANSION DETAILS")
print("=" * 80)

for i, step in enumerate(steps, start=1):
    print(
        f"{i}. {step['city']:<15} "
        f"g(n)={step['g(n)']:<5} "
        f"h(n)={step['h(n)']:<5} "
        f"f(n)={step['f(n)']:<5} "
        f"path={' -> '.join(step['path'])}"
    )



A* EXPANSION DETAILS
1. Seattle         g(n)=0     h(n)=150   f(n)=150   path=Seattle
2. San Francisco   g(n)=678   h(n)=200   f(n)=878   path=Seattle -> San Francisco
3. Los Angeles     g(n)=1026  h(n)=150   f(n)=1176  path=Seattle -> San Francisco -> Los Angeles
4. Riverside       g(n)=1064  h(n)=150   f(n)=1214  path=Seattle -> San Francisco -> Riverside
5. Phoenix         g(n)=1371  h(n)=100   f(n)=1471  path=Seattle -> San Francisco -> Riverside -> Phoenix
6. Chicago         g(n)=1737  h(n)=100   f(n)=1837  path=Seattle -> Chicago
7. Detroit         g(n)=1975  h(n)=100   f(n)=2075  path=Seattle -> Chicago -> Detroit
8. Dallas          g(n)=2258  h(n)=100   f(n)=2358  path=Seattle -> San Francisco -> Riverside -> Phoenix -> Dallas
9. Atlanta         g(n)=2325  h(n)=50    f(n)=2375  path=Seattle -> Chicago -> Atlanta
10. Washington      g(n)=2371  h(n)=50    f(n)=2421  path=Seattle -> Chicago -> Detroit -> Washington
11. Houston         g(n)=2386  h(n)=50    f(n)=2436  path=Seattle